## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Step 1: Data Loading & Cleaning

### Loading Dataset

In [ ]:
df = pd.read_csv('Dataset.csv')
print(f'Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

### Cleaning Column Names

In [ ]:
df.columns = df.columns.str.strip()

### Checking Missing Values

In [ ]:
missing_count = df.isnull().sum()
missing_count

### Checking & Removing Duplicates

In [ ]:
duplicate_count = df.duplicated().sum()
print(f'Duplicates Found: {duplicate_count}')
if duplicate_count > 0:
    df = df.drop_duplicates()
    print('Duplicates removed.')

### Standardizing Categorical Columns

In [ ]:
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].str.strip()

df.dtypes

## Step 2: Exploratory Data Analysis (EDA)

### Descriptive Statistics

In [ ]:
df.describe().T

### Revenue Dependency — Promotion Baseline

In [ ]:
promo_revenue = df.groupby('Promo Code Used')['Purchase Amount (USD)'].sum()
promo_counts = df['Promo Code Used'].value_counts()

for status in ['Yes', 'No']:
    rev_pct = (promo_revenue[status] / df['Purchase Amount (USD)'].sum()) * 100
    cnt_pct = (promo_counts[status] / len(df)) * 100
    print(f'Promo Used: {status} → {rev_pct:.2f}% of Revenue ({cnt_pct:.2f}% of Transactions)')

### Distribution of Purchase Amount

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Purchase Amount (USD)'], kde=True, color='teal')
plt.title('Distribution of Purchase Amount (USD)')
plt.tight_layout()
plt.show()

### Order Counts by Product Category

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x='Category', data=df, palette='Set2')
plt.title('Order Counts by Product Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Step 3: Feature Engineering

### Value Tier — Monetary Value Proxy

Formula: `Purchase Amount × (Previous Purchases + 1)` estimates total historical customer value. Segmented at 30th and 75th percentile to form Low / Medium / High tiers.

In [ ]:
df['Total_Estimated_Spend'] = df['Purchase Amount (USD)'] * (df['Previous Purchases'] + 1)

q30 = df['Total_Estimated_Spend'].quantile(0.30)
q75 = df['Total_Estimated_Spend'].quantile(0.75)

def segment_value_tier(spend):
    if spend <= q30: return 'Low-Value'
    elif spend <= q75: return 'Medium-Value'
    else: return 'High-Value'

df['value_tier'] = df['Total_Estimated_Spend'].apply(segment_value_tier)
df['value_tier'].value_counts()

### Promo Dependency Score (0–3)

Measures reliance on discount behavior. Score 0 = fully organic, Score 3 = fully promo-dependent.

In [ ]:
def evaluate_promo_dependency(row):
    score = 0
    if row['Promo Code Used'] == 'Yes' or row['Discount Applied'] == 'Yes':
        score += 2
    if row['Subscription Status'] == 'No' and score > 0:
        if row['Frequency of Purchases'] in ['Weekly', 'Bi-Weekly', 'Fortnightly', 'Monthly']:
            score += 1
    return score

df['dependency_score'] = df.apply(evaluate_promo_dependency, axis=1)
df['dependency_score'].value_counts()

### Satisfaction Flag — Brand Health Proxy

Review Rating ≥ 4.0 → satisfied customer (flag = 1). Acts as a proxy for brand promoters.

In [ ]:
df['satisfaction_flag'] = np.where(df['Review Rating'] >= 4.0, 1, 0)
df[['value_tier', 'dependency_score', 'satisfaction_flag']].sample(5)

## Step 4: Loyalty Definition Faceoff

### Definition A — Frequency & Organic Driven

Customers with `Previous Purchases > 15` who did **not** use a promo code. Captures genuine repeat buyers with no discount dependency.

In [ ]:
df['Loyalty_Def_A'] = np.where(
    (df['Previous Purchases'] > 15) & (df['Promo Code Used'] == 'No'), 1, 0
)
print(f'Definition A — Organic Loyalists: {df["Loyalty_Def_A"].sum()}')

### Definition B — Value & Engagement Driven

Customers in the `High-Value` tier **or** active subscribers. Captures monetary value regardless of promo usage.

In [ ]:
df['Loyalty_Def_B'] = np.where(
    (df['value_tier'] == 'High-Value') | (df['Subscription Status'] == 'Yes'), 1, 0
)
print(f'Definition B — Monetary Loyalists: {df["Loyalty_Def_B"].sum()}')

### Statistical Comparison — Which Definition Wins?

In [ ]:
metrics_comparison = {
    'Metric': ['Avg Estimated Spend', 'Avg Review Rating', 'Avg Dependency Score'],
    'Definition A (Organic)': [
        df[df['Loyalty_Def_A'] == 1]['Total_Estimated_Spend'].mean(),
        df[df['Loyalty_Def_A'] == 1]['Review Rating'].mean(),
        df[df['Loyalty_Def_A'] == 1]['dependency_score'].mean()
    ],
    'Definition B (Monetary)': [
        df[df['Loyalty_Def_B'] == 1]['Total_Estimated_Spend'].mean(),
        df[df['Loyalty_Def_B'] == 1]['Review Rating'].mean(),
        df[df['Loyalty_Def_B'] == 1]['dependency_score'].mean()
    ]
}

pd.DataFrame(metrics_comparison)

### Winner: Definition A

Def A chosen because it filters out promo-driven buyers and correlates with genuine brand affinity. Def B's higher count (1,735 vs 1,529) is misleading — it includes discount-dependent buyers whose retention is not brand-driven.

## Step 5: Export Cleaned & Engineered Dataset

In [ ]:
df.to_csv('engineered_customer_data.csv', index=False)
print('Saved as engineered_customer_data.csv')